# Bank Manager: evaluate, improve, control, deploy

The agent stays local during evaluation. Microsoft Foundry supplies the configured model deployments for inference, test generation, and judging; hosting happens only after the optimized agent passes its gates.

1. validate the local Agent Framework adapter;
2. run the reviewed 19-case Toolkit quality matrix;
3. inspect and repair one failed Toolkit case;
4. run ASSERT on the original baseline and inspect what fails;
5. replace only the prompt, rerun the same frozen ASSERT cases, and measure the improvement;
6. explain why instructions remain probabilistic and add deterministic ACS/ECS controls to the prompt-optimized agent;
7. run ASSERT on the combined prompt + ACS/ECS agent and decide whether it is ready;
8. optionally inspect the retained ACS-only ablation or compare another Foundry model;
9. deploy exactly the evaluated prompt + ACS/ECS agent as a hosted agent.

Every agent variant has a stable callable, ASSERT configuration, and run ID in `bank_manager/agent_variants.yaml`, so the audience can find each version after the demo.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd()
if ROOT.name != "ASSERT-bank-manager-demo":
    candidate = ROOT / "ASSERT-bank-manager-demo"
    if candidate.exists():
        ROOT = candidate

load_dotenv(ROOT / ".env")

PYTHON = Path(sys.executable)
FOUNDRY_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT") or os.getenv("AZURE_API_BASE")
PRIMARY_TARGET = os.getenv(
    "AZURE_AI_MODEL_DEPLOYMENT_NAME",
    os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME", "gpt-5.6-terra"),
)
EVALUATOR_TARGET = os.getenv("AZURE_AI_EVALUATOR_MODEL_DEPLOYMENT_NAME", PRIMARY_TARGET)
TOOLKIT_EVALUATOR_TARGET = os.getenv(
    "AZURE_AI_TOOLKIT_EVALUATOR_MODEL_DEPLOYMENT_NAME",
    EVALUATOR_TARGET,
)
ALTERNATE_TARGET = os.getenv("BANK_MANAGER_ALTERNATE_MODEL_DEPLOYMENT_NAME", "gpt-5.6-luna")

if not FOUNDRY_ENDPOINT:
    raise RuntimeError("Set AZURE_OPENAI_ENDPOINT or AZURE_API_BASE to your Microsoft Foundry endpoint.")

FOUNDRY_ENV = {
    "AZURE_OPENAI_ENDPOINT": FOUNDRY_ENDPOINT,
    "AZURE_API_BASE": FOUNDRY_ENDPOINT,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": PRIMARY_TARGET,
    "AZURE_AI_EVALUATOR_MODEL_DEPLOYMENT_NAME": EVALUATOR_TARGET,
}


def run(
    command: list[str],
    *,
    env_overrides: dict[str, str] | None = None,
) -> subprocess.CompletedProcess[str]:
    environment = os.environ.copy()
    environment.update(FOUNDRY_ENV)
    environment.update(env_overrides or {})
    environment["PYTHONUNBUFFERED"] = "1"
    environment["PYTHONIOENCODING"] = "utf-8"
    environment["PYTHONUTF8"] = "1"

    print(">", " ".join(command), flush=True)
    for variable in (
        "AZURE_AI_MODEL_DEPLOYMENT_NAME",
        "AZURE_AI_TOOLKIT_EVALUATOR_MODEL_DEPLOYMENT_NAME",
        "AZURE_AI_EVALUATOR_MODEL_DEPLOYMENT_NAME",
    ):
        if variable in environment:
            print(f"  {variable}={environment[variable]}", flush=True)

    started_at = time.monotonic()
    process = subprocess.Popen(
        command,
        cwd=ROOT,
        env=environment,
        text=True,
        encoding="utf-8",
        errors="replace",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    output_lines: list[str] = []
    assert process.stdout is not None
    for line in process.stdout:
        output_lines.append(line)
        print(line, end="", flush=True)
    returncode = process.wait()
    elapsed = time.monotonic() - started_at
    print(f"\nCompleted with return code {returncode} in {elapsed:.1f}s", flush=True)
    return subprocess.CompletedProcess(
        command,
        returncode,
        stdout="".join(output_lines),
        stderr=None,
    )


print(f"Root: {ROOT}")
print(f"Python: {PYTHON}")
print(f"Foundry endpoint: {FOUNDRY_ENDPOINT}")
print(f"Primary deployment: {PRIMARY_TARGET}")
print(f"Toolkit evaluator deployment: {TOOLKIT_EVALUATOR_TARGET}")
print(f"ASSERT evaluator deployment: {EVALUATOR_TARGET}")
print(f"Alternate deployment: {ALTERNATE_TARGET}")

## 1. Validate the local adapter

This check does not call a model. It constructs the local `BankManagerAgent` and verifies that all 13 tool schemas expose names, descriptions, and parameters to the Toolkit evaluators.

In [ ]:
sys.path.insert(0, str(ROOT))

from bank_manager.toolkit_local_agent import build_bank_manager_agent

if not (os.getenv("AZURE_OPENAI_ENDPOINT") or os.getenv("AZURE_API_BASE")):
    os.environ["AZURE_API_BASE"] = "https://schema-check.invalid"

agent = build_bank_manager_agent()
schemas = [tool.to_json_schema_spec() for tool in agent.default_options["tools"]]
schemas = [schema.get("function", schema) for schema in schemas]

assert len(schemas) == 13
assert all(schema.get("name") and schema.get("description") and "parameters" in schema for schema in schemas)
print([schema["name"] for schema in schemas])

## 2. Run the complete Toolkit evaluation and review the scorecard

The first code block keeps the evaluation visible: it runs all 19 reviewed cases against the local Agent Framework agent with six evaluators, producing 114 checks and a fresh Toolkit result artifact.

The second code block recovers transient evaluator infrastructure errors before building the scorecard. Recovery reruns all six evaluators only for affected cases so every replacement row remains internally consistent. Connection, timeout, throttling, and service failures use bounded retries; valid evaluator failures are never retried or converted to passes. Exhausted or non-transient errors remain visible and stop scorecard generation.

`pytest-agent-evals` does not support GPT-5 reasoning models as judges. Set `AZURE_AI_TOOLKIT_EVALUATOR_MODEL_DEPLOYMENT_NAME` to a deployed supported chat judge such as `gpt-4.1`; the later ASSERT evaluations can continue using the reasoning-aware `AZURE_AI_EVALUATOR_MODEL_DEPLOYMENT_NAME`.

Review the scorecard before continuing. Choose one failed case to investigate in step 3; a failed quality gate still produces the complete scorecard and row-level evidence.

In [ ]:
# Prepare the result directory and remember existing artifacts so this run's output can be identified.
results_dir = ROOT / "test-results"
result_pattern = "evaluation_results_*.json"
results_dir.mkdir(exist_ok=True)

before_toolkit_run = {
    path.resolve(): path.stat().st_mtime_ns
    for path in results_dir.glob(result_pattern)
}

# -u disables Python buffering; -s disables pytest capture; -vv prints every evaluator/case node ID.
toolkit_command = [
    str(PYTHON),
    "-u",
    "-m",
    "pytest",
    "toolkit-evaluation/test_bank_manager_toolkit.py",
    "--agent-module",
    "bank_manager.agents.baseline.agent",
    "-vv",
    "-s",
    "--tb=short",
    "--durations=10",
    "--cache-clear",
    "--cache-mode=session",
]

toolkit_result = run(toolkit_command)

# Find the fresh result file written by this run.
fresh_toolkit_results = [
    path
    for path in results_dir.glob(result_pattern)
    if path.resolve() not in before_toolkit_run
    or path.stat().st_mtime_ns > before_toolkit_run[path.resolve()]
]
if not fresh_toolkit_results:
    raise RuntimeError(
        "The Toolkit run did not produce a fresh evaluation result artifact. "
        f"Pytest return code: {toolkit_result.returncode}"
    )

toolkit_result_path = max(
    fresh_toolkit_results,
    key=lambda path: path.stat().st_mtime_ns,
)
print(f"Toolkit result: {toolkit_result_path}")
print(f"Pytest return code: {toolkit_result.returncode}")

In [ ]:
import importlib

from scripts import render_foundry_report as render_foundry_report_module
from scripts import toolkit_scorecard as toolkit_scorecard_module

# Reload the renderer first so the scorecard wrapper imports its latest implementation.
render_foundry_report_module = importlib.reload(render_foundry_report_module)
toolkit_scorecard_module = importlib.reload(toolkit_scorecard_module)

# Repair only transient evaluator errors, replacing complete six-evaluator rows.
toolkit_result_path, toolkit_scorecard_returncode = (
    toolkit_scorecard_module.recover_toolkit_result(
        result_path=toolkit_result_path,
        root=ROOT,
        agent_deployment=PRIMARY_TARGET,
        evaluator_deployment=TOOLKIT_EVALUATOR_TARGET,
        pytest_returncode=toolkit_result.returncode,
        python_executable=PYTHON,
        agent_revision="baseline",
    )
)

# Convert the complete Toolkit artifact into the notebook scorecard.
toolkit_rows = toolkit_scorecard_module.display_toolkit_scorecard(
    result_path=toolkit_result_path,
    root=ROOT,
    agent_deployment=PRIMARY_TARGET,
    evaluator_deployment=TOOLKIT_EVALUATOR_TARGET,
    pytest_returncode=toolkit_scorecard_returncode,
)


## 3. Inspect the selected failure

Inspect the failed `conflicting-admin-request / tool_call_accuracy` row as one ordered visual: user input, tool calls and results, final response, and evaluator verdict.

In [ ]:
import importlib

from IPython.display import HTML, display

from scripts.inspect_toolkit_results import evaluator_names, inspect_row
from scripts import render_toolkit_trace as trace_renderer

# Reload the renderer so notebook reruns always pick up the latest visual code.
render_toolkit_trace = importlib.reload(trace_renderer).render_toolkit_trace

CASE_ID = "trade-standard-conditional"
EVALUATOR = "tool_call_accuracy"

selected_row = next(
    (row for row in toolkit_rows if row.get("inputs.id") == CASE_ID),
    None,
)
if selected_row is None:
    available_ids = [row.get("inputs.id") for row in toolkit_rows]
    raise ValueError(f"Unknown CASE_ID {CASE_ID!r}. Choose from: {available_ids}")
if EVALUATOR not in evaluator_names(selected_row):
    raise ValueError(f"{EVALUATOR!r} is not available for {CASE_ID!r}")

selected_trace = inspect_row(selected_row)
display(HTML(render_toolkit_trace(selected_trace, EVALUATOR)))

pytest_args = [
    "toolkit-evaluation/test_bank_manager_toolkit.py",
    "-k",
    f"{EVALUATOR} and {CASE_ID}",
    "-v",
    "--cache-clear",
    "--cache-mode=session",
]
focused_command = [str(PYTHON), "-m", "pytest", *pytest_args]


## 4. ASSERT baseline: establish the problem before changing the agent

ASSERT goes beyond the single-turn Toolkit checks. It generates multi-turn prompt and scenario cases, captures trajectories and tool use, and judges both impermissible behavior and regressions on legitimate work.

Start with the original agent. `baseline.yaml` generates the taxonomy and the frozen 60-case suite, then evaluates the complete `bank_manager.agents.baseline.agent:chat` package. Let it reach `Pipeline completed`.

Every later variant reuses those exact cases, tester settings, judge dimensions, and model deployment. As changing the generated cases would invalidate the experiment.

The cell finishes by starting the local ASSERT viewer and displaying a direct link to this baseline run, so you can inspect its conversations, traces, tool calls, and judgments before continuing.

In [ ]:
# Load the baseline ASSERT config directly for its suite and run IDs.
import yaml
from IPython.display import display

baseline_config_path = ROOT / "bank_manager" / "evals" / "baseline.yaml"
baseline_config = yaml.safe_load(baseline_config_path.read_text(encoding="utf-8"))

from urllib.error import URLError
from urllib.request import urlopen
import shutil


# Helper: True if the ASSERT viewer web app answers at the given URL.
def assert_viewer_is_ready(url: str) -> bool:
    try:
        with urlopen(url, timeout=1) as response:
            return response.status < 500
    except (OSError, URLError):
        return False


# THE ASSERT RUN. One command executes the four stages configured in baseline.yaml:
#   1. systematize - turn the broad behavior description into a taxonomy of allowed/not-allowed behaviors
#   2. test_set    - generate the frozen 60-case suite (40 prompt + 20 scenario cases)
#   3. inference   - a tester model converses (max 5 turns) with the local baseline agent; tool calls are traced
#   4. judge       - judge models score each transcript (policy_violation, overrefusal, ungrounded_policy_claim)
print("\nRun ASSERT baseline: generate taxonomy + frozen 60-case suite, infer, and judge")

baseline_result = run([
    str(PYTHON),
    "-m",
    "assert_ai.cli",
    "run",
    "--config",
    str(baseline_config_path),
])

# Read the suite and run IDs so the viewer can open this run directly.
baseline_suite = baseline_config["suite"]
baseline_run_id = baseline_config["run"]

# Start the viewer only when it is not already running.
viewer_path = ROOT / "viewer"
assert_dashboard_url = "http://127.0.0.1:5174"
if not assert_viewer_is_ready(assert_dashboard_url):
    npm = shutil.which("npm")
    if not npm:
        raise RuntimeError("Node.js/npm is required to start the ASSERT dashboard")
    if not (viewer_path / "node_modules").is_dir():
        install_viewer = run([npm, "--prefix", str(viewer_path), "ci"])
        if install_viewer.returncode != 0:
            raise RuntimeError("ASSERT viewer dependency installation failed")
    ASSERT_VIEWER_PROCESS = subprocess.Popen(
        [npm, "run", "dev"],
        cwd=viewer_path,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
    )
    deadline = time.monotonic() + 90
    while time.monotonic() < deadline and not assert_viewer_is_ready(assert_dashboard_url):
        if ASSERT_VIEWER_PROCESS.poll() is not None:
            raise RuntimeError("ASSERT dashboard exited during startup")
        time.sleep(0.25)
    if not assert_viewer_is_ready(assert_dashboard_url):
        raise RuntimeError("ASSERT dashboard did not become ready")

# Build and display the direct dashboard link.
baseline_dashboard_url = (
    f"{assert_dashboard_url}/suite/{baseline_suite}/{baseline_run_id}"
)
print(f"Baseline dashboard: {baseline_dashboard_url}")
display(
    HTML(
        f'<p><a href="{baseline_dashboard_url}" target="_blank" rel="noopener">'
        "<strong>Open the first ASSERT baseline run</strong></a></p>"
    )
)


## 5. Run ASSERT on the stage 2 ASSERT-optimized agent

`assert_optimized_instructions.yaml` points to `bank_manager.agents.assert_optimized_instructions.agent:chat`. That package is the baseline plus ASSERT-driven instruction and tool-description improvements, without ACS/ECS enforcement.

Proceed directly from Step 4.1 to this step. The baseline taxonomy and all 60 cases remain frozen. This is the clean prompt-only comparison.

In [ ]:
# Reload the catalog so this step does not reuse stale variants from an earlier kernel session.
import yaml

variant_catalog_path = ROOT / "bank_manager" / "agent_variants.yaml"
variant_catalog = yaml.safe_load(variant_catalog_path.read_text(encoding="utf-8"))["variants"]

# Shared result-reading helpers for this and later comparisons.
SUITE_DIR = ROOT / "artifacts" / "results" / "bank-manager-feature-rep"
ASSERT_RUNS = {
    key: variant["run_id"]
    for key, variant in variant_catalog.items()
}


def load_jsonl(path: Path) -> list[dict]:
    if not path.is_file():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def summarize_assert_run(label: str, *, show_examples: bool = False) -> dict[str, float]:
    run_id = ASSERT_RUNS[label]
    scores_path = SUITE_DIR / run_id / "scores.jsonl"
    rows = [row for row in load_jsonl(scores_path) if row.get("judge_status") == "ok"]
    if not rows:
        print(f"{label:13} | no completed scores at {scores_path}")
        return {}

    dimensions = sorted({
        name
        for row in rows
        for name in row.get("verdict", {}).get("dimensions", {})
    })
    rates = {}
    for dimension in dimensions:
        applicable = [
            row.get("verdict", {}).get("dimensions", {}).get(dimension)
            for row in rows
            if dimension in row.get("verdict", {}).get("dimensions", {})
        ]
        rates[dimension] = sum(value is True for value in applicable) / len(applicable)

    rate_text = ", ".join(f"{name}={rate:.1%}" for name, rate in rates.items())
    print(f"{label:13} | {len(rows):2} judged cases | {rate_text}")

    if show_examples:
        failures = [
            row for row in rows
            if any(value is True for value in row.get("verdict", {}).get("dimensions", {}).values())
        ]
        for row in failures[:3]:
            verdict = row.get("verdict", {})
            print(f"\n{row.get('test_case_id')} ({row.get('type')}):")
            print(verdict.get("justification", "No justification recorded."))
    return rates


prompt_result = run([
    str(PYTHON), "-m", "assert_ai.cli", "run", "--config",
    "bank_manager/evals/assert_optimized_instructions.yaml",
])
if prompt_result.returncode != 0:
    raise RuntimeError(f"ASSERT stage 2 failed with return code {prompt_result.returncode}")

print("\nBaseline versus ASSERT-optimized prompt (lower violation rates are better)")
baseline_rates = summarize_assert_run("baseline")
assert_optimized_instructions_rates = summarize_assert_run("assert_optimized_instructions", show_examples=True)

## 6. Prompting improves intent; ACS enforces boundaries

The prompt-only result shows what better instructions can achieve, but the model still interprets those instructions probabilistically. A prompt cannot guarantee that every future model response follows a transfer sequence, refuses an admin bypass, protects a sensitive result, or ignores an instruction embedded in retrieved content.

The production candidate therefore keeps the optimized prompt and adds ACS/ECS with OPA at deterministic intervention points:

- agent input and output are evaluated by `control.run(...)`;
- all 13 tools are wrapped by `control.run_tool(...)`;
- pre/post snapshots carry typed facts such as `risk_tier`, referenced entities, approval state, admin state, and grounding;
- Rego gates deny sensitive reads/actions, approval bypasses, admin elevation, out-of-scope references, and ungrounded policy claims;
- allowed standard-tier and grounded flows continue normally.

This is additive: stage 3 `assert_optimized_instructions_acs` freezes the same full prompt and tool definitions from stage 2, then layers its revision-local ACS/ECS manifest and policy around them.

In [ ]:
import re
import shutil
import yaml

# Refresh run IDs so new ACS results are not compared using stale cached mappings.
variant_catalog = yaml.safe_load((ROOT / "bank_manager" / "agent_variants.yaml").read_text(encoding="utf-8"))["variants"]
ASSERT_RUNS = {key: variant["run_id"] for key, variant in variant_catalog.items()}

from bank_manager.agents.assert_optimized_instructions_acs import agent as assert_optimized_instructions_acs_agent
from bank_manager.agents.assert_optimized_instructions_acs.agent import ACS_MANIFEST

assert_optimized_instructions_acs_agent._ensure_opa_on_path()
policy_path = ACS_MANIFEST.parent / "policy" / "bank_manager_feature.rego"
policy_source = policy_path.read_text(encoding="utf-8")
policy_reasons = sorted(set(re.findall(r'"([a-z_]+_gate)"', policy_source)))

print("Stage 3 target: ASSERT-optimized prompt + ACS/ECS")
print("Manifest:", ACS_MANIFEST)
print("Policy:", policy_path)
print("Deterministic gates:")
for reason in policy_reasons:
    print(" -", reason)

opa = shutil.which("opa")
if not opa:
    raise RuntimeError("OPA 1.8+ is required for the combined prompt + ACS/ECS agent")
policy_test_result = run(
    [
        opa,
        "test",
        "bank_manager/agents/assert_optimized_instructions_acs/policy/bank_manager_feature.rego",
        "bank_manager/agents/assert_optimized_instructions_acs/policy/bank_manager_feature_test.rego",
        "-v",
    ]
)
if policy_test_result.returncode != 0:
    raise RuntimeError(f"ACS/ECS policy tests failed with return code {policy_test_result.returncode}")
acs_result = run([
    str(PYTHON), "-m", "assert_ai.cli", "run", "--config",
    "bank_manager/evals/assert_optimized_instructions_acs.yaml",
])
if acs_result.returncode != 0:
    raise RuntimeError(f"ASSERT stage 3 failed with return code {acs_result.returncode}")

print("\nThree-step outcome (lower violation rates are better)")
for arm in ("baseline", "assert_optimized_instructions", "assert_optimized_instructions_acs"):
    summarize_assert_run(arm, show_examples=(arm == "assert_optimized_instructions_acs"))

## 7. Test an alternate Foundry model on the improved local agent

This is a controlled model comparison. The edited instructions, tools, frozen ASSERT cases, ACS policy, tester, judge, and thresholds stay fixed. Only the target model changes from the configured primary deployment to the configured alternate deployment.

Run the next code cell once to perform a fresh paid Toolkit evaluation on `cheaper_model`, recover incomplete cases, display its separate scorecard, and resume frozen ASSERT with its completeness check and model comparison. The adapter uses the existing LangGraph + MCP + ACS runner on `gpt-5.6-luna`. ASSERT reuses completed work when configuration is unchanged; recovering an inference can invalidate the judge cache and rejudge the suite. Quality failures are reported, not retried. The other agents and their scorecards are unchanged. The following review cells are optional and make no model calls.

In [ ]:
# Result validation needed by the alternate-model comparison below.
from assert_ai.config import load_config


def load_baseline_assert_reference() -> tuple[int, str]:
    """Use the completed baseline run as the comparison reference."""
    baseline_dir = SUITE_DIR / ASSERT_RUNS["baseline"]
    manifest_path = baseline_dir / "manifest.json"
    if not manifest_path.is_file():
        raise RuntimeError(f"baseline: missing {manifest_path}")

    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    scores = load_jsonl(baseline_dir / "scores.jsonl")
    if not scores or any(row.get("judge_status") != "ok" for row in scores):
        raise RuntimeError("baseline: every generated case must have a successful judge outcome")

    test_set_hash = (
        manifest.get("artifact_versions", {})
        .get("test_set", {})
        .get("file_hashes", {})
        .get("test_set")
    )
    if not test_set_hash:
        raise RuntimeError("baseline: manifest does not record the generated test-set hash")
    return len(scores), test_set_hash


def require_complete_assert_arm(label: str) -> dict:
    expected_cases, expected_test_set_hash = load_baseline_assert_reference()
    run_dir = SUITE_DIR / ASSERT_RUNS[label]
    manifest_path = run_dir / "manifest.json"
    if not manifest_path.is_file():
        raise RuntimeError(f"{label}: missing {manifest_path}")

    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("status") != "completed":
        raise RuntimeError(f"{label}: run status is {manifest.get('status')!r}, expected 'completed'")
    for stage in ("inference", "judge"):
        if manifest.get("stages", {}).get(stage) != "completed":
            raise RuntimeError(f"{label}: {stage} stage is not complete")

    scores = load_jsonl(run_dir / "scores.jsonl")
    successful_scores = [row for row in scores if row.get("judge_status") == "ok"]
    incomplete_scores = [row for row in scores if row.get("judge_status") != "ok"]
    if len(scores) != expected_cases or len(successful_scores) != expected_cases:
        failure_details = "\n".join(
            f"- {row.get('test_case_id', 'unknown')}: "
            f"{row.get('judge_status', 'missing')} — {row.get('judge_error') or 'no reason recorded'}"
            for row in incomplete_scores
        )
        config_path = variant_catalog[label]["assert_config"]
        raise RuntimeError(
            f"{label}: found {len(successful_scores)} successful judge outcomes "
            f"across {len(scores)} score rows; expected {expected_cases}.\n"
            f"Failed or skipped cases:\n{failure_details or '- score rows are missing'}\n"
            f"Rerun `{config_path}`. ASSERT now automatically retries failed judge rows "
            "and inference rows refused by the provider."
        )

    test_set = manifest.get("artifact_versions", {}).get("test_set", {})
    observed_hash = test_set.get("file_hashes", {}).get("test_set")
    if observed_hash != expected_test_set_hash:
        raise RuntimeError(f"{label}: frozen test-set hash mismatch")

    saved_config = load_config(run_dir / "config.yaml")
    trace = saved_config["pipeline"]["inference"]["target"].get("trace", {})
    if trace.get("backend") != "otel":
        raise RuntimeError(f"{label}: saved run did not use OTel tracing")

    print(f"{label:13} | complete | {len(scores)} scores | OTel | test set {observed_hash[:12]}...")
    return manifest


from bank_manager.agents.cheaper_model.agent import DEFAULT_DEPLOYMENT as CHEAPER_DEPLOYMENT

if ALTERNATE_TARGET != CHEAPER_DEPLOYMENT:
    raise RuntimeError(f"cheaper_model is pinned to {CHEAPER_DEPLOYMENT}, not {ALTERNATE_TARGET}")

cheaper_results_dir = ROOT / "test-results"
cheaper_results_dir.mkdir(exist_ok=True)
before_cheaper_toolkit = {
    path.resolve(): path.stat().st_mtime_ns
    for path in cheaper_results_dir.glob("evaluation_results_*.json")
}
cheaper_toolkit_command = [
    str(PYTHON),
    "-m",
    "pytest",
    "toolkit-evaluation/test_bank_manager_toolkit.py",
"--agent-module",
"bank_manager.agents.cheaper_model.agent",
"-v",
    "--cache-clear",
    "--cache-mode=session",
]

alternate_toolkit_result = run(
    cheaper_toolkit_command,
    env_overrides={
        "AZURE_AI_MODEL_DEPLOYMENT_NAME": ALTERNATE_TARGET,
        "AZURE_AI_TOOLKIT_EVALUATOR_MODEL_DEPLOYMENT_NAME": TOOLKIT_EVALUATOR_TARGET,
    },
)
fresh_cheaper_results = [
    path for path in cheaper_results_dir.glob("evaluation_results_*.json")
    if path.resolve() not in before_cheaper_toolkit
    or path.stat().st_mtime_ns > before_cheaper_toolkit[path.resolve()]
]
if not fresh_cheaper_results:
    raise RuntimeError(f"No fresh cheaper-model Toolkit artifact; return code {alternate_toolkit_result.returncode}")
cheaper_toolkit_result_path = max(fresh_cheaper_results, key=lambda path: path.stat().st_mtime_ns)
cheaper_toolkit_returncode = alternate_toolkit_result.returncode
print(f"Saved Toolkit artifact: {cheaper_toolkit_result_path}; return code {cheaper_toolkit_returncode}")

def review_cheaper_toolkit_results(*, retry_incomplete=False):
    import importlib
    from scripts import toolkit_scorecard
    toolkit_scorecard = importlib.reload(toolkit_scorecard)
    global cheaper_toolkit_result_path, cheaper_toolkit_returncode
    rows = toolkit_scorecard.load_rows(cheaper_toolkit_result_path)
    incomplete_cases = [
        row["inputs.id"] for row in rows
        if any(row.get(f"outputs.{name}.result") not in {"pass", "fail"}
               for name in toolkit_scorecard.TOOLKIT_EVALUATORS)
    ]
    if incomplete_cases:
        print("Incomplete/error Toolkit cases:", incomplete_cases)
        if not retry_incomplete:
            raise RuntimeError("Toolkit scorecard is incomplete; rerun the review with retry_incomplete=True.")
        cheaper_toolkit_result_path, cheaper_toolkit_returncode = toolkit_scorecard.recover_toolkit_result(
            result_path=cheaper_toolkit_result_path, root=ROOT,
            agent_deployment=CHEAPER_DEPLOYMENT, evaluator_deployment=TOOLKIT_EVALUATOR_TARGET,
            pytest_returncode=cheaper_toolkit_returncode, python_executable=PYTHON,
            incomplete_case_ids=incomplete_cases, agent_revision="cheaper_model",
        )
    result = toolkit_scorecard.display_toolkit_scorecard(
        result_path=cheaper_toolkit_result_path, root=ROOT,
        agent_deployment=CHEAPER_DEPLOYMENT, evaluator_deployment=TOOLKIT_EVALUATOR_TARGET,
        pytest_returncode=cheaper_toolkit_returncode,
        scorecard_path=cheaper_results_dir / "toolkit-cheaper-model-acs-scorecard.json",
    )
    if cheaper_toolkit_returncode != 0:
        raise RuntimeError("Toolkit execution returned a nonzero status; inspect the output before ASSERT")
    return result

# This cell intentionally makes paid Toolkit calls, including incomplete-case recovery.
cheaper_toolkit_rows = review_cheaper_toolkit_results(retry_incomplete=True)

# ASSERT resumes the same run; a changed inference set can invalidate the judge cache.
cheaper_assert_result = run([
    str(PYTHON), "-m", "assert_ai.cli", "run", "--config",
    "bank_manager/evals/cheaper_model.yaml",
])
if cheaper_assert_result.returncode != 0:
    raise RuntimeError(
        f"ASSERT alternate-model arm failed with return code {cheaper_assert_result.returncode}"
    )
require_complete_assert_arm("cheaper_model")
model_comparison = {
    label: summarize_assert_run(label)
    for label in ("assert_optimized_instructions_acs", "cheaper_model")
}
print("\nInterpret this as a model comparison, not an intervention comparison.")
print("The prompt, ACS/ECS policy, tools, cases, tester, judge, and thresholds are fixed.")


Optional: review the saved ASSERT results again without executing evaluations. Step 7 already runs the cheaper `gpt-5.6-luna` package using `cheaper_model.yaml`. That config omits `systematize` and `test_set`, so it cannot regenerate the comparison artifacts.

In [ ]:
# Review the saved scorecard without running paid evaluations or recovery.
cheaper_toolkit_rows = review_cheaper_toolkit_results()

## 9. Deploy the optimized prompt + ACS/ECS agent as a hosted agent

Everything above this point runs locally. This final step is the first operation that creates a hosted-agent version. The deployment path is explicit:

`azure.yaml` → `hosted_agent_main.py` → `bank_manager.hosted_agent.main` → `bank_manager.agents.assert_optimized_instructions_acs.agent.run_async`

That is the same stage 3 implementation evaluated as `variant-s4-acs-optimized-gpt56terra`. Every local version remains discoverable in `bank_manager/agent_variants.yaml`, with a stable callable and evaluation metadata; ASSERT artifacts remain under `artifacts/results/bank-manager-feature-rep/<run-id>`.

Before opting in, review the model and case-level ASSERT evidence. The first code cell explicitly defines the final agent revision, module, source path, and model deployment. The deployment cell then:

1. verifies that the hosted adapter imports the combined runner;
2. downloads pinned Linux OPA 1.8.0 and verifies its SHA-256 checksum; at runtime the agent copies this bundled binary from the read-only package into writable temporary storage before executing it;
3. sets the selected model deployment in the current `azd` environment;
4. packages and deploys `bank-manager-agent` as a new immutable version;
5. confirms that the hosted version is active;
6. sends one read-only smoke-test request through the Responses endpoint with bounded cold-start retries, and prints the hosted session log automatically if all attempts fail.

Equivalent commands:

```powershell
pwsh -NoProfile -File scripts/prepare_hosted_opa.ps1
azd env set AZURE_AI_MODEL_DEPLOYMENT_NAME <FINAL_MODEL_DEPLOYMENT>
azd deploy bank-manager-agent --no-prompt
azd ai agent show bank-manager-agent --output json
azd ai agent invoke bank-manager-agent "Summarize the transfer approval steps without performing a transfer."
azd ai agent monitor bank-manager-agent --tail 100  # only if invocation fails
```

Review the final-agent configuration before running the deployment cell.

In [ ]:
# Explicitly select the evaluated agent implementation and model to deploy.
FINAL_AGENT_REVISION = "assert_optimized_instructions_acs"
FINAL_AGENT_MODULE = "bank_manager.agents.assert_optimized_instructions_acs.agent"
FINAL_AGENT_PATH = ROOT / "bank_manager" / "agents" / "assert_optimized_instructions_acs" / "agent.py"
FINAL_MODEL_DEPLOYMENT = PRIMARY_TARGET

print(f"Final agent revision: {FINAL_AGENT_REVISION}")
print(f"Final agent module: {FINAL_AGENT_MODULE}")
print(f"Final agent path: {FINAL_AGENT_PATH}")
print(f"Final model deployment: {FINAL_MODEL_DEPLOYMENT}")

In [ ]:
SELECTED_TARGET = FINAL_MODEL_DEPLOYMENT
SELECTED_EVALUATED_ARM = FINAL_AGENT_REVISION
if SELECTED_EVALUATED_ARM not in ASSERT_RUNS:
    raise RuntimeError(f"Final agent revision {SELECTED_EVALUATED_ARM!r} has no ASSERT run")

hosted_adapter_path = ROOT / "bank_manager" / "hosted_agent" / "main.py"
hosted_adapter_source = hosted_adapter_path.read_text(encoding="utf-8")
revision_source = FINAL_AGENT_PATH.read_text(encoding="utf-8")
acs_shim_source = (FINAL_AGENT_PATH.parent / "acs_shim.py").read_text(encoding="utf-8")
assert FINAL_AGENT_MODULE in hosted_adapter_source, (
    "Hosted adapter is not wired to the evaluated prompt + ACS/ECS runner"
)
assert 'PACKAGE_DIR / "bin" / "opa"' in revision_source and "shutil.copyfile" in revision_source, (
    "Hosted OPA must be copied from the read-only package into writable temporary storage"
)
assert "_opa_executable()" in acs_shim_source, (
    "ACS shim must resolve OPA after hosted runtime setup"
)
assert variant_catalog[FINAL_AGENT_REVISION]["hosted_entrypoint"] == "bank_manager/hosted_agent/main.py"
print("Verified hosted chain:")
print(" azure.yaml -> hosted_agent_main.py -> bank_manager.hosted_agent.main -> assert_optimized_instructions_acs.run_async")
print(f" Evaluated arm: {SELECTED_EVALUATED_ARM}")
print(f" Evaluated run: {ASSERT_RUNS[SELECTED_EVALUATED_ARM]}")
print(f" Selected model: {SELECTED_TARGET}")
print(" Hosted implementation: optimized prompt + ACS/ECS")

DEPLOY_HOSTED_AGENT = True

if not DEPLOY_HOSTED_AGENT:
    raise RuntimeError(
        "Deployment is disabled. Review SELECTED_TARGET and combined ASSERT evidence, "
        "then set DEPLOY_HOSTED_AGENT = True when you are ready."
    )

azd_environment = {"AZURE_DEV_USER_AGENT": "microsoft_foundry_skill"}

prepare_opa = run([
    "pwsh",
    "-NoProfile",
    "-File",
    "scripts/prepare_hosted_opa.ps1",
])
assert prepare_opa.returncode == 0, "Failed to prepare the checksum-verified hosted OPA runtime"

set_model = run(
    [
        "azd",
        "env",
        "set",
        "AZURE_AI_MODEL_DEPLOYMENT_NAME",
        SELECTED_TARGET,
    ],
    env_overrides=azd_environment,
)
assert set_model.returncode == 0, "Failed to set the selected model in the azd environment"

deploy = run(
    [
        "azd",
        "deploy",
        "bank-manager-agent",
        "--no-prompt",
    ],
    env_overrides=azd_environment,
)
assert deploy.returncode == 0, "Hosted-agent deployment failed"

show = run(
    [
        "azd",
        "ai",
        "agent",
        "show",
        "bank-manager-agent",
        "--output",
        "json",
    ],
    env_overrides=azd_environment,
)
assert show.returncode == 0, "The deployed agent did not report an active status"

smoke_command = [
    "azd",
    "ai",
    "agent",
    "invoke",
    "bank-manager-agent",
    "Summarize the transfer approval steps without performing a transfer.",
]
smoke_test = None
for attempt in range(1, 4):
    print(f"Hosted smoke-test attempt {attempt}/3")
    smoke_test = run(smoke_command, env_overrides=azd_environment)
    if smoke_test.returncode == 0:
        break
    if attempt < 3:
        delay_seconds = attempt * 10
        print(f"Smoke test failed during startup; retrying in {delay_seconds}s...")
        time.sleep(delay_seconds)

assert smoke_test is not None
if smoke_test.returncode != 0:
    print("\nHosted smoke test failed after 3 attempts. Recent session logs:")
    run(
        ["azd", "ai", "agent", "monitor", "bank-manager-agent", "--tail", "100"],
        env_overrides=azd_environment,
    )
    raise RuntimeError("Hosted-agent smoke test failed after 3 attempts; inspect the logs above")

print("Hosted-agent smoke test passed.")